# 🎨 BigQuery Anti-Pattern Recognition - Streamlit Frontend

This notebook demonstrates how to deploy and use a Streamlit web application for the BigQuery Anti-Pattern Recognition tool. The frontend provides an intuitive interface for:

1. **Query Input** - Easy text box for pasting SQL queries
2. **Analysis Display** - Visual representation of detected anti-patterns
3. **AI Rewriting** - Automated query optimization suggestions
4. **Performance Visualizations** - Charts showing improvement potential

## 📋 Prerequisites

Before running this notebook, ensure you have completed:
- ✅ **Notebook 1**: Setup and deployment
- ✅ **Notebook 2**: Cloud Run API testing
- ✅ **Notebook 3**: BigQuery UDF testing

## 🚀 Quick Start

Run all cells in this notebook to:
1. Verify your configuration
2. Test the Streamlit app locally
3. Deploy to Cloud Run (optional)
4. Access your web-based anti-pattern analyzer

In [ ]:
# Import required libraries
import os
import json
import subprocess
import time
import webbrowser
from IPython.display import display, HTML, Markdown
import ipywidgets as widgets
from utils import ConfigManager, DeploymentHelper

print("📦 Libraries imported successfully!")
print("🎨 Ready to deploy Streamlit frontend!")

## 🔧 Configuration Verification

Let's verify that your configuration from previous notebooks is available:

In [ ]:
# Load and verify configuration
config_manager = ConfigManager()
config = config_manager.load_config()

if config:
    print("✅ Configuration loaded successfully!")
    print("\n📋 Current Configuration:")
    print(f"   🏗️  Project ID: {config.get('project_id', 'Not set')}")
    print(f"   📊 Dataset ID: {config.get('dataset_id', 'Not set')}")
    print(f"   🔗 Connection ID: {config.get('connection_id', 'Not set')}")
    print(f"   ☁️  Cloud Run URL: {config.get('cloud_run_url', 'Not set')}")
    print(f"   🏷️  Service Name: {config.get('service_name', 'Not set')}")
    
    # Check if all required components are configured
    required_fields = ['project_id', 'dataset_id', 'cloud_run_url']
    missing_fields = [field for field in required_fields if not config.get(field)]
    
    if missing_fields:
        print(f"\n⚠️  Missing configuration: {', '.join(missing_fields)}")
        print("   Please run the setup notebook (01_setup_and_deploy.ipynb) first.")
    else:
        print("\n🎉 All required configuration is present!")
        
else:
    print("❌ No configuration found!")
    print("   Please run the setup notebook (01_setup_and_deploy.ipynb) first.")

## 🎨 Streamlit Application Overview

The Streamlit application (`streamlit_app.py`) provides a user-friendly web interface with the following features:

### 🏗️ Application Architecture

```
┌─────────────────────────────────────────────────────────────┐
│                    Streamlit Frontend                       │
├─────────────────────────────────────────────────────────────┤
│  📝 Query Input     │  📊 Analysis Results                  │
│  • Text area        │  • Anti-pattern detection            │
│  • Sample queries   │  • Severity classification           │
│  • Load/Clear       │  • Detailed recommendations          │
├─────────────────────────────────────────────────────────────┤
│  🤖 AI Rewriting    │  📈 Visualizations                   │
│  • Query optimization│  • Performance impact charts        │
│  • Before/after     │  • Cost estimation                   │
│  • Improvement list │  • Severity distribution             │
└─────────────────────────────────────────────────────────────┘
```

### 🎯 Key Components

1. **📝 Query Input Section**
   - Large text area for SQL queries
   - Sample query selector from sidebar
   - Analyze and clear buttons

2. **📊 Analysis Results Section**
   - Summary metrics (anti-patterns found, severity, processing time)
   - Detailed anti-pattern descriptions and recommendations
   - Color-coded severity indicators

3. **🤖 AI-Powered Query Rewriting**
   - Automated query optimization
   - Side-by-side comparison (original vs. rewritten)
   - List of improvements made

4. **📈 Performance Visualizations**
   - Anti-pattern severity distribution (pie chart)
   - Performance impact estimation (bar chart)
   - Cost comparison (before/after optimization)

## 🖥️ Local Development Server

Let's start the Streamlit application locally for testing:

In [ ]:
# Create interactive controls for local server
start_button = widgets.Button(
    description="🚀 Start Local Server",
    button_style='success',
    layout=widgets.Layout(width='200px')
)

stop_button = widgets.Button(
    description="🛑 Stop Server",
    button_style='danger',
    layout=widgets.Layout(width='200px')
)

open_browser_button = widgets.Button(
    description="🌐 Open in Browser",
    button_style='info',
    layout=widgets.Layout(width='200px')
)

output_area = widgets.Output()

# Global variable to track server process
streamlit_process = None

def start_streamlit_server(b):
    global streamlit_process
    with output_area:
        output_area.clear_output()
        print("🚀 Starting Streamlit server...")
        
        try:
            # Start Streamlit in the background
            streamlit_process = subprocess.Popen(
                ['streamlit', 'run', 'streamlit_app.py', '--server.port=8501', '--server.headless=true'],
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                text=True
            )
            
            # Wait a moment for server to start
            time.sleep(3)
            
            print("✅ Streamlit server started successfully!")
            print("📍 Local URL: http://localhost:8501")
            print("\n🎯 Features available:")
            print("   • Query input and analysis")
            print("   • Sample query selection")
            print("   • AI-powered query rewriting")
            print("   • Performance visualizations")
            print("   • Analysis history tracking")
            
        except Exception as e:
            print(f"❌ Failed to start server: {str(e)}")
            print("\n🔧 Troubleshooting:")
            print("   1. Ensure Streamlit is installed: pip install streamlit")
            print("   2. Check if port 8501 is available")
            print("   3. Verify streamlit_app.py exists in current directory")

def stop_streamlit_server(b):
    global streamlit_process
    with output_area:
        output_area.clear_output()
        
        if streamlit_process:
            try:
                streamlit_process.terminate()
                streamlit_process.wait(timeout=5)
                print("🛑 Streamlit server stopped successfully!")
                streamlit_process = None
            except Exception as e:
                print(f"⚠️  Error stopping server: {str(e)}")
                print("   You may need to manually stop the process.")
        else:
            print("ℹ️  No server process found to stop.")

def open_in_browser(b):
    with output_area:
        print("🌐 Opening Streamlit app in browser...")
        try:
            webbrowser.open('http://localhost:8501')
            print("✅ Browser opened successfully!")
        except Exception as e:
            print(f"❌ Failed to open browser: {str(e)}")
            print("   Please manually navigate to: http://localhost:8501")

# Attach event handlers
start_button.on_click(start_streamlit_server)
stop_button.on_click(stop_streamlit_server)
open_browser_button.on_click(open_in_browser)

# Display controls
print("🎮 Streamlit Server Controls:")
display(widgets.HBox([start_button, stop_button, open_browser_button]))
display(output_area)

## 🧪 Testing the Application

Once your Streamlit server is running, you can test the application with these sample scenarios:

In [ ]:
# Display testing scenarios
testing_scenarios = [
    {
        "name": "SELECT * Anti-pattern",
        "query": "SELECT * FROM `project.dataset.large_table` WHERE date > '2023-01-01'",
        "expected": "Should detect SELECT_STAR anti-pattern"
    },
    {
        "name": "ORDER BY without LIMIT",
        "query": "SELECT name, score FROM `project.dataset.scores` ORDER BY score DESC",
        "expected": "Should detect ORDER_BY_WITHOUT_LIMIT anti-pattern"
    },
    {
        "name": "REGEXP_CONTAINS misuse",
        "query": "SELECT * FROM `project.dataset.users` WHERE REGEXP_CONTAINS(email, 'gmail')",
        "expected": "Should detect REGEXP_CONTAINS anti-pattern"
    },
    {
        "name": "Clean query",
        "query": "SELECT user_id, email FROM `project.dataset.users` WHERE created_date >= '2023-01-01' LIMIT 1000",
        "expected": "Should show no anti-patterns detected"
    }
]

print("🧪 Testing Scenarios for Streamlit App:")
print("\nCopy and paste these queries into the Streamlit interface to test different scenarios:\n")

for i, scenario in enumerate(testing_scenarios, 1):
    print(f"📋 **Scenario {i}: {scenario['name']}**")
    print(f"   Query: `{scenario['query']}`")
    print(f"   Expected: {scenario['expected']}")
    print()

print("🎯 **Testing Checklist:**")
print("   ✅ Query input accepts SQL text")
print("   ✅ Sample queries load from sidebar")
print("   ✅ Analysis results display correctly")
print("   ✅ Anti-patterns are color-coded by severity")
print("   ✅ AI rewriting generates optimized queries")
print("   ✅ Visualizations render properly")
print("   ✅ Analysis history is tracked")
print("   ✅ Configuration status shows in sidebar")

## ☁️ Cloud Deployment (Optional)

Deploy your Streamlit application to Cloud Run for production use:

In [ ]:
# Cloud deployment configuration
deployment_config = {
    'service_name': 'bq-antipattern-frontend',
    'region': 'us-central1',
    'memory': '1Gi',
    'cpu': '1',
    'max_instances': '10',
    'port': '8501'
}

deploy_button = widgets.Button(
    description="☁️ Deploy to Cloud Run",
    button_style='primary',
    layout=widgets.Layout(width='250px')
)

deployment_output = widgets.Output()

def deploy_to_cloud_run(b):
    with deployment_output:
        deployment_output.clear_output()
        print("☁️ Starting Cloud Run deployment...")
        
        if not config:
            print("❌ Configuration not found. Please run setup notebook first.")
            return
        
        project_id = config.get('project_id')
        if not project_id:
            print("❌ Project ID not found in configuration.")
            return
        
        try:
            print("📝 Creating Dockerfile for Streamlit app...")
            
            # Create Dockerfile for Streamlit
            dockerfile_content = f'''FROM python:3.9-slim

WORKDIR /app

# Copy requirements and install dependencies
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy application files
COPY streamlit_app.py .
COPY utils.py .
COPY config.json .

# Expose port
EXPOSE {deployment_config['port']}

# Health check
HEALTHCHECK CMD curl --fail http://localhost:{deployment_config['port']}/_stcore/health

# Run Streamlit
CMD ["streamlit", "run", "streamlit_app.py", "--server.port={deployment_config['port']}", "--server.address=0.0.0.0"]
'''
            
            with open('Dockerfile.streamlit', 'w') as f:
                f.write(dockerfile_content)
            
            print("✅ Dockerfile created successfully!")
            
            # Build and deploy
            service_name = deployment_config['service_name']
            region = deployment_config['region']
            
            print(f"🏗️ Building container image...")
            build_cmd = f"gcloud builds submit --tag gcr.io/{project_id}/{service_name}-frontend ."
            
            print(f"📦 Deploying to Cloud Run...")
            deploy_cmd = f'''gcloud run deploy {service_name}-frontend \
                --image gcr.io/{project_id}/{service_name}-frontend \
                --platform managed \
                --region {region} \
                --allow-unauthenticated \
                --memory {deployment_config['memory']} \
                --cpu {deployment_config['cpu']} \
                --max-instances {deployment_config['max_instances']} \
                --port {deployment_config['port']}'''
            
            print("\n🔧 **Manual Deployment Steps:**")
            print("\n1. Build the container:")
            print(f"   {build_cmd}")
            print("\n2. Deploy to Cloud Run:")
            print(f"   {deploy_cmd}")
            
            print("\n📋 **Deployment Configuration:**")
            for key, value in deployment_config.items():
                print(f"   {key}: {value}")
            
            print("\n🎉 After deployment, your Streamlit app will be available at:")
            print(f"   https://{service_name}-frontend-<hash>-{region}.a.run.app")
            
        except Exception as e:
            print(f"❌ Deployment preparation failed: {str(e)}")

deploy_button.on_click(deploy_to_cloud_run)

print("☁️ Cloud Run Deployment:")
display(deploy_button)
display(deployment_output)

## 🔒 Security and Access Control

Configure security settings for your Streamlit application:

In [ ]:
# Security configuration options
print("🔒 Security Configuration Options:")
print()

security_options = [
    {
        "option": "Public Access (Default)",
        "description": "Anyone with the URL can access the application",
        "command": "--allow-unauthenticated",
        "use_case": "Demos, public tools, testing"
    },
    {
        "option": "Authenticated Access",
        "description": "Requires Google Cloud authentication",
        "command": "--no-allow-unauthenticated",
        "use_case": "Internal tools, sensitive data"
    },
    {
        "option": "IAM-based Access",
        "description": "Fine-grained access control with IAM roles",
        "command": "gcloud run services add-iam-policy-binding",
        "use_case": "Enterprise deployments"
    }
]

for i, option in enumerate(security_options, 1):
    print(f"**{i}. {option['option']}**")
    print(f"   Description: {option['description']}")
    print(f"   Command: `{option['command']}`")
    print(f"   Use case: {option['use_case']}")
    print()

print("🛡️ **Additional Security Recommendations:**")
print("   • Enable Cloud Armor for DDoS protection")
print("   • Use Cloud Load Balancer with SSL certificates")
print("   • Implement rate limiting for API calls")
print("   • Monitor access logs with Cloud Logging")
print("   • Set up alerting for unusual activity")

## 📊 Monitoring and Analytics

Set up monitoring for your Streamlit application:

In [ ]:
# Monitoring setup
print("📊 Monitoring and Analytics Setup:")
print()

monitoring_components = [
    {
        "component": "Cloud Monitoring",
        "metrics": ["Request count", "Response time", "Error rate", "Memory usage"],
        "setup": "Automatically enabled for Cloud Run services"
    },
    {
        "component": "Cloud Logging",
        "metrics": ["Application logs", "Access logs", "Error logs", "Audit logs"],
        "setup": "Configure log levels in Streamlit app"
    },
    {
        "component": "Cloud Trace",
        "metrics": ["Request tracing", "Performance analysis", "Bottleneck identification"],
        "setup": "Enable tracing in application code"
    },
    {
        "component": "Custom Analytics",
        "metrics": ["Query analysis count", "Anti-pattern detection rate", "User engagement"],
        "setup": "Implement custom metrics in Streamlit app"
    }
]

for component in monitoring_components:
    print(f"**{component['component']}**")
    print(f"   Metrics: {', '.join(component['metrics'])}")
    print(f"   Setup: {component['setup']}")
    print()

print("📈 **Key Performance Indicators (KPIs):**")
kpis = [
    "Daily active users",
    "Queries analyzed per day",
    "Anti-patterns detected per query",
    "Average response time",
    "User session duration",
    "Error rate percentage"
]

for kpi in kpis:
    print(f"   • {kpi}")

print("\n🚨 **Alerting Recommendations:**")
alerts = [
    "High error rate (>5%)",
    "Slow response time (>10s)",
    "Memory usage spike (>80%)",
    "Unusual traffic patterns",
    "Service downtime"
]

for alert in alerts:
    print(f"   • {alert}")

## 🎯 Usage Examples and Best Practices

Here are some practical examples of how to use the Streamlit frontend effectively:

In [ ]:
# Usage examples and best practices
print("🎯 Streamlit Frontend Usage Examples:")
print()

use_cases = [
    {
        "title": "Developer Code Review",
        "scenario": "A developer wants to check their SQL query before submitting a pull request",
        "steps": [
            "Paste the SQL query into the text area",
            "Click 'Analyze Query' to detect anti-patterns",
            "Review the detailed recommendations",
            "Use 'AI Rewrite' to get optimization suggestions",
            "Copy the optimized query back to their code"
        ]
    },
    {
        "title": "Executive Demo",
        "scenario": "Demonstrating the tool's capabilities to stakeholders",
        "steps": [
            "Use sample queries from the sidebar",
            "Show the visual analysis results",
            "Highlight cost savings potential",
            "Demonstrate the AI rewriting feature",
            "Show performance impact visualizations"
        ]
    },
    {
        "title": "Batch Query Analysis",
        "scenario": "Analyzing multiple queries from a data pipeline",
        "steps": [
            "Analyze queries one by one",
            "Review analysis history in sidebar",
            "Compare anti-pattern trends",
            "Export results for reporting",
            "Create improvement action plan"
        ]
    },
    {
        "title": "Training and Education",
        "scenario": "Teaching SQL best practices to a development team",
        "steps": [
            "Start with sample queries showing common mistakes",
            "Explain each anti-pattern and its impact",
            "Show the AI rewriting process",
            "Discuss the performance improvements",
            "Encourage team members to test their own queries"
        ]
    }
]

for i, use_case in enumerate(use_cases, 1):
    print(f"**{i}. {use_case['title']}**")
    print(f"   Scenario: {use_case['scenario']}")
    print("   Steps:")
    for step in use_case['steps']:
        print(f"      • {step}")
    print()

print("💡 **Best Practices:**")
best_practices = [
    "Always test queries with sample data first",
    "Review anti-pattern recommendations carefully",
    "Use AI rewriting as a starting point, not final solution",
    "Consider the specific context of your data and use case",
    "Monitor performance improvements in production",
    "Share findings with your team for collective learning"
]

for practice in best_practices:
    print(f"   • {practice}")

## 🎉 Summary and Next Steps

Congratulations! You have successfully set up the complete BigQuery Anti-Pattern Recognition demo package.

In [ ]:
# Final summary and next steps
print("🎉 Demo Package Complete!")
print()
print("📦 **What You've Built:**")
components = [
    "✅ Cloud Run API service for anti-pattern detection",
    "✅ BigQuery UDF for direct SQL integration",
    "✅ Interactive Jupyter notebooks for testing",
    "✅ Streamlit web application for user-friendly access",
    "✅ Comprehensive documentation and examples"
]

for component in components:
    print(f"   {component}")

print("\n🚀 **Next Steps:**")
next_steps = [
    "Test the Streamlit application with your own SQL queries",
    "Deploy the frontend to Cloud Run for team access",
    "Integrate the API into your existing development workflow",
    "Set up monitoring and alerting for production use",
    "Train your team on using the anti-pattern detection tools",
    "Customize the tool for your specific use cases"
]

for i, step in enumerate(next_steps, 1):
    print(f"   {i}. {step}")

print("\n📚 **Additional Resources:**")
resources = [
    "BigQuery best practices documentation",
    "SQL optimization guides",
    "Cloud Run deployment tutorials",
    "Streamlit application development",
    "Google Cloud monitoring and logging"
]

for resource in resources:
    print(f"   • {resource}")

print("\n🎯 **Success Metrics to Track:**")
metrics = [
    "Number of queries analyzed",
    "Anti-patterns detected and fixed",
    "Query performance improvements",
    "Cost savings achieved",
    "Developer adoption rate",
    "Time saved in code reviews"
]

for metric in metrics:
    print(f"   • {metric}")

print("\n🌟 **Thank you for using the BigQuery Anti-Pattern Recognition tool!**")
print("   Happy querying! 🚀")